# Lab Report 4: Portfolio Construction

**Ethan Wang, Kevin Yang**  
**RSM338**  
**March 9, 2026**

## 1 From Prices to Returns

### 1.1 Data Preparation
This analysis requires three datasets: DGS10.csv, market.csv, and prices.csv. We will load them in and perform some data cleaning to make sure that the date values are aligned.  
First, we load in the csv data. Then, we will then filter the data for only the last 10 years. We will handle missing data by dropping stocks that have over 10% of data missing. For the remaining missing values, we will forward fill using ffill(). We can now align the three datasets by date. Finally, we convert the risk-free data from an annualized percentage to daily log return.  
At the end, we should have data for stock prices, market index, and risk free rate all aligned to the same date index.

In [59]:
import pandas as pd
import numpy as np

# Stock prices
prices = pd.read_csv("prices.csv", parse_dates=["Date"], index_col="Date")

# Market index
market = pd.read_csv("market.csv", parse_dates=["Date"], index_col="Date")

# Risk-free rate
rf = pd.read_csv("DGS10.csv", parse_dates=["observation_date"], index_col="observation_date")
rf.index.name = "Date"

print("Prices:", prices.shape)
print("Market:", market.shape)
print("Risk-free:", rf.shape)

Prices: (13912, 503)
Market: (13912, 1)
Risk-free: (16727, 1)


In [60]:
# Filter for only the last 10 years
# Determine date range
end_date = prices.index.max()
start_date = end_date - pd.DateOffset(years=10)

# Filter datasets
prices = prices.loc[start_date:end_date]
market = market.loc[start_date:end_date]
rf = rf.loc[start_date:end_date]

print("Filtered prices shape:", prices.shape)

Filtered prices shape: (2516, 503)


In [61]:
# Minimum number of required observations
threshold = int(0.9 * len(prices))

# Drop stocks with too many missing values
prices = prices.dropna(axis=1, thresh=threshold)

print("Remaining number of stocks:", prices.shape[1])

Remaining number of stocks: 476


In [62]:
# Forward fill remaining missing values
prices = prices.ffill()

In [63]:
# Find common trading dates
common_dates = prices.index.intersection(market.index).intersection(rf.index)

# Align datasets
prices = prices.loc[common_dates]
market = market.loc[common_dates]
rf = rf.loc[common_dates]

print("Final number of observations:", len(common_dates))

Final number of observations: 2516


In [64]:
# Convert risk-free rate from annual percentage to daily log returns
rf_daily = np.log(1 + rf["DGS10"]/100) / 252
rf_daily = rf_daily.to_frame(name="rf")

rf_daily.head() 

,rf
Date,
2015-03-04,0.000083
2015-03-05,0.000083
2015-03-06,0.000088
2015-03-09,0.000086
2015-03-10,0.000084


In [65]:
prices.head()

,A,AAPL,ABBV,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP,...,WTW,WY,WYNN,XEL,XOM,XYL,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2015-03-04,38.679150,28.706537,39.771614,38.854362,18.811941,77.067551,77.629997,47.521145,35.235287,69.662910,...,107.532234,23.238997,121.084457,25.248211,56.308872,31.467518,47.678905,106.658325,91.029999,42.945576
2015-03-05,38.881775,28.230852,37.521389,39.209774,18.989443,78.033951,78.620003,47.561821,35.534653,69.799713,...,108.143448,23.144056,121.241493,25.503759,56.024681,31.370514,47.940876,107.035538,91.160004,43.159248
2015-03-06,38.246311,28.273281,36.716324,38.432858,18.976765,76.864090,77.550003,47.138695,34.614117,69.196228,...,106.875664,22.398127,119.496986,24.569178,55.307724,31.009026,47.131145,104.493889,89.629997,42.527561
2015-03-09,38.439720,28.393879,36.650341,38.705601,19.179623,76.965813,77.930000,47.569962,34.718884,69.872101,...,107.622734,22.743965,115.868446,24.883141,55.004173,31.079567,47.297848,105.032761,89.400002,43.001328
2015-03-10,37.417477,27.806528,36.848297,38.160099,18.954576,74.651497,76.010002,46.544689,34.067768,68.512344,...,106.649300,22.540529,112.963898,24.941561,54.422886,30.594639,46.369045,103.739487,87.800003,42.453228


In [66]:
market.tail()

,^GSPC
Date,
2025-02-26,5956.060059
2025-02-27,5861.569824
2025-02-28,5954.500000
2025-03-03,5849.720215
2025-03-04,5778.149902


In [67]:
print("Final dataset summary")
print("Stock trading days:", prices.shape[0])
print("Market Index trading days:", market.shape[0])
print("Risk-free Rate trading days:", rf_daily.shape[0])
print("Stocks:", prices.shape[1])

Final dataset summary
Stock trading days: 2516
Market Index trading days: 2516
Risk-free Rate trading days: 2516
Stocks: 476


Taking a quick look at our cleaned datasets, we end up with 2516 trading days of data across 476 stocks.  

### 1.2 Daily Log Returns

### Problem 1a — Computing Log Returns
We compute daily log returns for each stock and for the market index using the formula:

$
r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)
$

where (P_t) represents the price at time (t) and (P_{t-1}) is the price on the previous trading day.

In [68]:
# Compute log returns
stock_returns = np.log(prices / prices.shift(1))
market_returns = np.log(market / market.shift(1))

# Remove only the first row created by shift
stock_returns = stock_returns.iloc[1:]
market_returns = market_returns.iloc[1:]

print(stock_returns.shape)
print(market_returns.shape)

(2515, 476)
(2515, 1)


In [69]:
rf_daily = rf_daily.loc[stock_returns.index]

print(rf_daily.shape)

(2515, 1)


**Gen using GPT--- MAKE MORE HUMAN**
Log returns are preferred in empirical finance for several reasons. First, they are time-additive, meaning that returns over longer horizons can be obtained by summing shorter-period log returns. This property is particularly useful later in the analysis when we aggregate daily returns into monthly returns for the CAPM regressions. Second, log returns tend to exhibit more convenient statistical properties, such as being closer to normally distributed, which simplifies many econometric and portfolio optimization procedures.